# Switch-only variant (winner_market != incumbent_market)
Restrict to elections with cabinet orientation switch and re-estimate RD validity and LP-IV.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.lpiv import first_stage, iv_estimate, reduced_form
from src.paths import ANALYSIS_DIR, PAPER_FIGURES_DIR, PAPER_LOGS_DIR, PAPER_TABLES_DIR
from src.rd import density_discontinuity, select_bandwidth
from src.rd_localrand import select_window_by_balance
from src.viz_style import savefig, set_style

In [2]:
set_style()

pos_path = ANALYSIS_DIR / "rd_event_panel_pos.parquet"
neg_path = ANALYSIS_DIR / "rd_event_panel_neg.parquet"

for path in [pos_path, neg_path]:
    if not path.exists():
        raise FileNotFoundError(f"Missing event panel: {path}")

panel_pos = pd.read_parquet(pos_path)
panel_neg = pd.read_parquet(neg_path)

# Switch-only filter
panel_pos = panel_pos[panel_pos["market_switch"] == 1].copy()
panel_neg = panel_neg[panel_neg["market_switch"] == 1].copy()

RUNNING = "running_var_vote"
HORIZONS = [0, 1, 2, 3, 4, 5]
controls = [
    "lag1_log_gdp_pc_const",
    "lag1_trade_open_gdp",
    "lag1_inflation_cpi_ann_pct",
    "lag1_efw_summary",
]

# Instruments
panel_pos["z_pos"] = (panel_pos[RUNNING] > 0).astype(int)
panel_neg["z_neg"] = (panel_neg[RUNNING] < 0).astype(int)

In [3]:
# Density diagnostics

def density_block(frame: pd.DataFrame, label: str) -> dict:
    series = frame[RUNNING].dropna()
    bandwidth = select_bandwidth(series, quantile=0.3, max_bw=0.1)
    if pd.isna(bandwidth):
        bandwidth = 0.05
    stats = density_discontinuity(series, bandwidth=bandwidth)
    stats["sample"] = label
    stats["bandwidth"] = bandwidth
    return stats


density_df = pd.DataFrame([
    density_block(panel_pos, "pos_switch"),
    density_block(panel_neg, "neg_switch"),
])
PAPER_TABLES_DIR.mkdir(parents=True, exist_ok=True)
density_path = PAPER_TABLES_DIR / "rd_density_switch.csv"
density_df.to_csv(density_path, index=False)

In [4]:
# Balance checks
balance_vars = [
    "lag1_log_gdp_pc_const",
    "lag1_trade_open_gdp",
    "lag1_inflation_cpi_ann_pct",
    "lag1_efw_summary",
    "lag1_inv_share_gdp",
]
windows = [0.01, 0.02, 0.03, 0.04, 0.05]

choice_pos, table_pos = select_window_by_balance(
    panel_pos,
    RUNNING,
    balance_vars,
    windows=windows,
    p_threshold=0.15,
    cluster="iso3c",
)
choice_neg, table_neg = select_window_by_balance(
    panel_neg,
    RUNNING,
    balance_vars,
    windows=windows,
    p_threshold=0.15,
    cluster="iso3c",
)

valid_pos_path = PAPER_TABLES_DIR / "rd_validity_switch_pos.csv"
valid_neg_path = PAPER_TABLES_DIR / "rd_validity_switch_neg.csv"

table_pos.to_csv(valid_pos_path, index=False)
table_neg.to_csv(valid_neg_path, index=False)

window_choice = {
    "positive": {
        "window": choice_pos.window,
        "p_threshold": choice_pos.p_threshold,
        "windows_tested": choice_pos.windows_tested,
    },
    "negative": {
        "window": choice_neg.window,
        "p_threshold": choice_neg.p_threshold,
        "windows_tested": choice_neg.windows_tested,
    },
}

PAPER_LOGS_DIR.mkdir(parents=True, exist_ok=True)
choice_path = PAPER_LOGS_DIR / "rd_window_choice_switch.json"
choice_path.write_text(json.dumps(window_choice, indent=2))

window_pos = choice_pos.window or 0.03
window_neg = choice_neg.window or 0.03

/Users/gabrielsaco/anaconda3/lib/python3.11/site-packages/statsmodels/regression/linear_model.py:1992: RuntimeWarning: divide by zero encountered in scalar divide
  self.het_scale = self.nobs/(self.df_resid)*(self.wresid**2)
/Users/gabrielsaco/anaconda3/lib/python3.11/site-packages/statsmodels/regression/linear_model.py:1992: RuntimeWarning: invalid value encountered in multiply
  self.het_scale = self.nobs/(self.df_resid)*(self.wresid**2)
/Users/gabrielsaco/anaconda3/lib/python3.11/site-packages/statsmodels/regression/linear_model.py:1992: RuntimeWarning: divide by zero encountered in scalar divide
  self.het_scale = self.nobs/(self.df_resid)*(self.wresid**2)
/Users/gabrielsaco/anaconda3/lib/python3.11/site-packages/statsmodels/regression/linear_model.py:1992: RuntimeWarning: invalid value encountered in multiply
  self.het_scale = self.nobs/(self.df_resid)*(self.wresid**2)
/Users/gabrielsaco/anaconda3/lib/python3.11/site-packages/statsmodels/regression/linear_model.py:1992: RuntimeWa

In [5]:
# Helper for rank failures

def safe_iv(frame: pd.DataFrame, **kwargs):
    try:
        return iv_estimate(frame, **kwargs)
    except ValueError:
        return iv_estimate(frame.iloc[0:0], **kwargs)


# First stage, reduced form, IV
fs_pos = first_stage(
    panel_pos,
    outcome_col="shock_efw",
    running_col=RUNNING,
    instrument_col="z_pos",
    controls=controls,
    window=window_pos,
    cluster="iso3c",
)
fs_neg = first_stage(
    panel_neg,
    outcome_col="shock_efw",
    running_col=RUNNING,
    instrument_col="z_neg",
    controls=controls,
    window=window_neg,
    cluster="iso3c",
)

fs_df = pd.DataFrame([
    {
        "sample": "pos",
        "outcome": "shock_efw",
        "coef": fs_pos.coef,
        "se": fs_pos.se,
        "pvalue": fs_pos.pvalue,
        "n_obs": fs_pos.n_obs,
        "window": fs_pos.window,
    },
    {
        "sample": "neg",
        "outcome": "shock_efw",
        "coef": fs_neg.coef,
        "se": fs_neg.se,
        "pvalue": fs_neg.pvalue,
        "n_obs": fs_neg.n_obs,
        "window": fs_neg.window,
    },
])

rf_rows = []
iv_rows = []
for h in HORIZONS:
    for sample_name, frame, instrument, window in [
        ("pos", panel_pos, "z_pos", window_pos),
        ("neg", panel_neg, "z_neg", window_neg),
    ]:
        rf = reduced_form(
            frame,
            outcome_col=f"log_gdp_cum_h{h}",
            running_col=RUNNING,
            instrument_col=instrument,
            controls=controls,
            window=window,
            cluster="iso3c",
        )
        rf_rows.append(
            {
                "sample": sample_name,
                "horizon": h,
                "coef": rf.coef,
                "se": rf.se,
                "pvalue": rf.pvalue,
                "n_obs": rf.n_obs,
                "window": rf.window,
            }
        )

        iv = safe_iv(
            frame,
            outcome_col=f"log_gdp_cum_h{h}",
            endog_col="shock_efw",
            running_col=RUNNING,
            instrument_col=instrument,
            controls=controls,
            window=window,
            cluster="iso3c",
        )
        iv_rows.append(
            {
                "sample": sample_name,
                "horizon": h,
                "coef": iv.coef,
                "se": iv.se,
                "pvalue": iv.pvalue,
                "n_obs": iv.n_obs,
                "window": iv.window,
            }
        )

rf_df = pd.DataFrame(rf_rows)
iv_df = pd.DataFrame(iv_rows)

PAPER_TABLES_DIR.mkdir(parents=True, exist_ok=True)
fs_df.to_csv(PAPER_TABLES_DIR / "irf_first_stage_switch.csv", index=False)
rf_df.to_csv(PAPER_TABLES_DIR / "irf_reduced_form_switch.csv", index=False)
iv_df.to_csv(PAPER_TABLES_DIR / "irf_iv_switch.csv", index=False)

/Users/gabrielsaco/anaconda3/lib/python3.11/site-packages/statsmodels/regression/linear_model.py:1992: RuntimeWarning: divide by zero encountered in scalar divide
  self.het_scale = self.nobs/(self.df_resid)*(self.wresid**2)
/Users/gabrielsaco/anaconda3/lib/python3.11/site-packages/statsmodels/regression/linear_model.py:1992: RuntimeWarning: divide by zero encountered in scalar divide
  self.het_scale = self.nobs/(self.df_resid)*(self.wresid**2)
/Users/gabrielsaco/anaconda3/lib/python3.11/site-packages/statsmodels/regression/linear_model.py:1992: RuntimeWarning: divide by zero encountered in scalar divide
  self.het_scale = self.nobs/(self.df_resid)*(self.wresid**2)
/Users/gabrielsaco/anaconda3/lib/python3.11/site-packages/statsmodels/regression/linear_model.py:1992: RuntimeWarning: divide by zero encountered in scalar divide
  self.het_scale = self.nobs/(self.df_resid)*(self.wresid**2)
/Users/gabrielsaco/anaconda3/lib/python3.11/site-packages/statsmodels/regression/linear_model.py:199

In [6]:
# Plot IRFs

def plot_irf(sample: str, color: str, path_suffix: str) -> None:
    subset = iv_df[iv_df["sample"] == sample].dropna(subset=["horizon", "coef"]).sort_values("horizon")
    fig, ax = plt.subplots(figsize=(6, 3))
    ax.plot(subset["horizon"], subset["coef"], marker="o", color=color, label=sample)
    ax.fill_between(
        subset["horizon"],
        subset["coef"] - 1.96 * subset["se"],
        subset["coef"] + 1.96 * subset["se"],
        color=color,
        alpha=0.2,
    )
    ax.axhline(0, color="black", linewidth=1)
    ax.set_xlabel("Horizon (years)")
    ax.set_ylabel("Log GDP per capita (cum)")
    ax.set_title(f"Switch-only IV IRF ({sample})")
    ax.legend(frameon=False)
    savefig(fig, PAPER_FIGURES_DIR / "irfs_switch" / path_suffix)
    plt.close(fig)


plot_irf("pos", "#2A9D8F", "irf_iv_pos_switch")
plot_irf("neg", "#E76F51", "irf_iv_neg_switch")

# Comparison
pos_plot = iv_df[iv_df["sample"] == "pos"].sort_values("horizon")
neg_plot = iv_df[iv_df["sample"] == "neg"].sort_values("horizon")
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(pos_plot["horizon"], pos_plot["coef"], marker="o", color="#2A9D8F", label="Positive")
ax.plot(neg_plot["horizon"], neg_plot["coef"], marker="o", color="#E76F51", label="Negative")
ax.axhline(0, color="black", linewidth=1)
ax.set_xlabel("Horizon (years)")
ax.set_ylabel("Log GDP per capita (cum)")
ax.set_title("Switch-only IV IRF comparison")
ax.legend(frameon=False)

savefig(fig, PAPER_FIGURES_DIR / "irfs_switch" / "irf_compare_switch")
plt.close(fig)

In [7]:
display(fs_df.style.set_caption("Switch-only first stage"))

,sample,outcome,coef,se,pvalue,n_obs,window
0,pos,shock_efw,2.541423,inf,1.000000,8,0.030000
1,neg,shock_efw,0.335372,0.243120,0.167757,12,0.050000
